# EgyptOra - Machine Learning Notebook

This notebook contains the complete ML workflow used for destination segmentation:
- Data preparation
- Feature Engineering
- Scaling
- PCA
- K-Means Clustering
- Model Evaluation
- Cluster Recommendations
- Similar Destination Analysis

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics.pairwise import pairwise_distances

print('Libraries imported successfully')

## 1. Load Data

The dataset represents each destination using tourism, accessibility and climate features.

In [ ]:
# Load your prepared destination dataset
# df = pd.read_excel('Destination_Features.xlsx')

df = pd.read_excel('Destination_Features.xlsx')
df.head()

## 2. Select Machine Learning Features

Features used to understand destination similarity:
- Number of attractions
- Category diversity
- Accessibility
- Attraction distribution
- Climate profile

In [ ]:
features = [
    'linked_attractions',
    'category_diversity',
    'avg_airport_distance_km',
    'avg_attraction_distance_km',
    'annual_avg_temp_c'
]

X = df[features]
X.head()

## 3. Feature Scaling

StandardScaler is used because features have different measurement ranges.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]

## 4. PCA Dimensionality Reduction

PCA keeps the majority of information while creating a compact representation.

In [ ]:
pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_scaled)

print('Number of components:', X_pca.shape[1])
print('Explained variance:', pca.explained_variance_ratio_.sum())

## 5. Test Different Cluster Numbers

We compare different values of K using Silhouette Score and Davies-Bouldin Index.

In [ ]:
results = []

for k in range(2,7):
    model = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = model.fit_predict(X_pca)

    results.append({
        'K': k,
        'Silhouette': silhouette_score(X_pca, labels),
        'Davies_Bouldin': davies_bouldin_score(X_pca, labels)
    })

evaluation = pd.DataFrame(results)
evaluation

## 6. Final K-Means Model

The selected model uses 4 clusters based on statistical quality and business interpretation.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=50)

df['cluster'] = kmeans.fit_predict(X_pca)

df[['destination_name','cluster']].head()

## 7. Business Cluster Names and Recommendations

In [ ]:
cluster_names = {
    0: 'Established High-Density Hubs',
    1: 'Compact Warm & Accessible',
    2: 'Dispersed Moderate-Access',
    3: 'Remote Low-Density / Access-Constrained'
}

recommendations = {
    0: 'Improve visitor experience and promote multi-attraction packages',
    1: 'Use seasonal campaigns and short tourism packages',
    2: 'Improve transportation routes and itinerary planning',
    3: 'Prioritize accessibility and infrastructure development'
}

df['cluster_name'] = df['cluster'].map(cluster_names)
df['recommendation'] = df['cluster'].map(recommendations)

df[['destination_name','cluster_name','recommendation']].head()

## 8. Similar Destination Engine

Find destinations with similar profiles using distance between standardized features.

In [ ]:
distance_matrix = pairwise_distances(X_scaled)

print('Similarity engine created successfully')

## Final Output

The ML output can be connected to Power BI to visualize:
- Destination clusters
- Cluster recommendations
- Similar destinations
- Decision support insights